<a href="https://colab.research.google.com/github/Fillo468/Emotion-Recognition-Multimodal-Fusion-/blob/main/REGRESSION_EMOVOME.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Import Dataset**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

from google.colab import drive
drive.mount('/content/drive')

# Uploading the preprocessed dataset
path = "/content/drive/MyDrive/EMOVOME_TopFeatures_Regression_Dataset.csv"
df = pd.read_csv(path)
df.columns = df.columns.str.strip()

Mounted at /content/drive


**Regression Computation**

In [ ]:
# Setting targets
y_arousal = df['arousal_num_C']
y_valence = df['valence_num_C']

# Normalizing EMOVOME output {-2, -1, 0, 1, 2} with respect to AMIGOS output [1, 9]
df['arousal_num_C'] = (df['arousal_num_C'] * 2) + 5
df['valence_num_C'] = (df['valence_num_C'] * 2) + 5

In [ ]:
# Selecting only the audio features
columns_to_remove = ['file_id', 'arousal_num_C', 'valence_num_C']
X = df.drop(columns=columns_to_remove, errors='ignore')

print(f"Final Dataset: {X.shape[0]} subjects, {X.shape[1]} features.\n")

Final Dataset: 999 subjects, 24 features.



In [ ]:
# Dividing the dataset into train, validation & test sets
X_temp, X_test, ya_temp, ya_test, yv_temp, yv_test = train_test_split(
    X, y_arousal, y_valence, test_size=0.2, random_state=42
)

X_train, X_val, ya_train, ya_val, yv_train, yv_val = train_test_split(
    X_temp, ya_temp, yv_temp, test_size=0.25, random_state=42
)

In [ ]:
# Normalizing audio features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Searching for best parameters
# parametri_rf = {
#     'n_estimators': [100, 300, 500],
#     'max_depth': [None, 10, 20],
#     'min_samples_split': [2, 5, 10],
#     'bootstrap': [True, False]
# }

# arousal_reg = GridSearchCV(RandomForestRegressor(random_state=42), parametri_rf, cv=5, n_jobs=-1) #Arousal: {'bootstrap': True, 'max_depth': None, 'min_samples_split': 5, 'n_estimators': 500}
# valence_reg = GridSearchCV(RandomForestRegressor(random_state=42), parametri_rf, cv=5, n_jobs=-1) #Valence: {'bootstrap': True, 'max_depth': 10, 'min_samples_split': 10, 'n_estimators': 100}

# miglior_modello_arousal = arousal_reg.best_estimator_
# miglior_modello_valence = valence_reg.best_estimator_

# print("\n--- MIGLIORI PARAMETRI RANDOM FOREST ---")
# print("Arousal:", arousal_reg.best_params_)
# print("Valence:", valence_reg.best_params_)

In [ ]:
# Creating regressors on valence & arousal
arousal_reg = RandomForestRegressor(n_estimators=500, max_depth=None, min_samples_split=5, bootstrap=True, random_state=42)
valence_reg = RandomForestRegressor(n_estimators=100, max_depth=10, min_samples_split=10, bootstrap=True, random_state=42)

# Saving the models for Late Fusion
joblib.dump(arousal_reg, 'modello_voce_arousal.pkl')
joblib.dump(valence_reg, 'modello_voce_valence.pkl')
print("Voice model saved")

# Training regressors
arousal_reg.fit(X_train_scaled, ya_train)
valence_reg.fit(X_train_scaled, yv_train)

Voice model saved


RandomForestRegressor(max_depth=10, min_samples_split=10, random_state=42)

In [ ]:
# Function to compute accuracy metrics
def accuracy_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    return {"MSE": mse, "RMSE": rmse, "MAE": mae, "R2": r2}

In [ ]:
# Computing regression on validation test
pred_a_val = arousal_reg.predict(X_val_scaled)
pred_v_val = valence_reg.predict(X_val_scaled)

met_val_a = accuracy_metrics(ya_val, pred_a_val)
met_val_v = accuracy_metrics(yv_val, pred_v_val)

# Computing regression on test set
pred_a_test = arousal_reg.predict(X_test_scaled)
pred_v_test = valence_reg.predict(X_test_scaled)

met_test_a = accuracy_metrics(ya_test, pred_a_test)
met_test_v = accuracy_metrics(yv_test, pred_v_test)

**Graphic Representation**

In [ ]:
metrics_table = [
    {"Phase": "Validation", "Target": "Arousal", **met_val_a},
    {"Phase": "Validation", "Target": "Valence", **met_val_v},
    {"Phase": "Test",       "Target": "Arousal", **met_test_a},
    {"Phase": "Test",       "Target": "Valence", **met_test_v}
]

df_metrics = pd.DataFrame(metrics_table)
df_metrics.set_index(["Phase", "Target"], inplace=True)
df_metrics = df_metrics.round(4)

display(df_metrics)

MSE    RMSE     MAE      R2
Phase      Target                                 
Validation Arousal  0.3880  0.6229  0.4751  0.1985
           Valence  0.9192  0.9588  0.7537  0.0453
Test       Arousal  0.3473  0.5893  0.4634  0.4331
           Valence  0.9822  0.9911  0.7570  0.0141